# QSTD — a look at the dataset

What is in the Quantum Sensor-aligned Telemetry Dataset, in six figures.

This notebook only *looks* at the data. It trains nothing: the models behind the
paper live in `experiments/`, are run by `scripts/reproduce_all.sh`, and their
numbers are read back in at the end. Keeping the two apart means a figure here
can never quietly disagree with a table there.

Run time: about two minutes.

    uv run python scripts/prepare_from_release.py data/qstd_v1.0.parquet
    uv run jupyter lab analysis/qstd_overview.ipynb

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA = REPO / "data" / "stage10_cleaned_final_dataset.parquet"
RESULTS = REPO / "experiments" / "results"

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7.5, 3.6),
                     "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})
SC, ION = "superconducting_20q", "trapped_ion_20q"
COLOUR = {SC: "#2b6cb0", ION: "#c05621"}
NAME = {SC: "superconducting, 20 qubits", ION: "trapped ion, 20 qubits"}

df = pd.read_parquet(DATA)
print(f"{len(df):,} records x {len(df.columns)} columns")
print("(the published dataset has 313 columns; prepare_from_release.py")
print(" restores the internal id and the split circuit id, giving 314 here)")

df["month"] = pd.to_datetime(df["completed_hour_utc"], utc=True).dt.tz_convert(None).dt.to_period("M")

## 1. What one record is

One record is one circuit execution: how it was submitted, what the circuit looked
like before and after transpilation, how far the measured outcomes fell from a
perfect run, and what the machine and the room were doing at the time.

In [ ]:
summary = pd.DataFrame({
    "records": df.groupby("device").size(),
    "jobs": df.groupby("device")["job_id"].nunique(),
    "first": df.groupby("device")["month"].min().astype(str),
    "last": df.groupby("device")["month"].max().astype(str),
}).sort_values("records", ascending=False)
summary["share"] = (100 * summary["records"] / len(df)).round(2).astype(str) + "%"
summary

## 2. Coverage is not uniform, and the dataset says so

Sensor and calibration data exist for part of the period, not all of it. Empty
means not measured — never zero. Any analysis that ignores this will read
missingness as signal.

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith(("sc_", "ion_")) and not c.startswith("sc_cal_")]
calib_cols = [c for c in df.columns if c.startswith("sc_cal_")]

cover = []
for dev in (SC, ION):
    d = df[df["device"] == dev]
    cover.append(pd.DataFrame({
        "month": d["month"].astype(str),
        "sensor": d[sensor_cols].notna().any(axis=1),
        "calibration": d[calib_cols].notna().any(axis=1) if calib_cols else False,
        "device": dev,
    }))
cover = pd.concat(cover).groupby(["device", "month"])[["sensor", "calibration"]].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, dev in zip(axes, (SC, ION)):
    c = cover.loc[dev]
    ax.bar(range(len(c)), c["sensor"], color=COLOUR[dev], label="sensor")
    ax.plot(range(len(c)), c["calibration"], color="#2d3748", lw=1.4,
            marker="o", ms=3, label="calibration")
    ax.set_xticks(range(0, len(c), 3))
    ax.set_xticklabels(c.index[::3], rotation=45, ha="right")
    ax.set_title(NAME[dev]); ax.set_ylim(0, 105)
axes[0].set_ylabel("% of records with data"); axes[0].legend(frameon=False)
fig.suptitle("Monthly coverage", y=1.02); fig.tight_layout()

## 3. The target

`hellinger_native` is the distance between the measured outcome distribution and
a noise-free simulation of the same circuit: 0 is a perfect run, 1 is no overlap.
The two devices sit in very different places, which is why the models are trained
per device.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
for dev in (SC, ION):
    v = df.loc[df["device"] == dev, "hellinger_native"].dropna()
    ax1.hist(v, bins=60, alpha=0.6, label=NAME[dev], color=COLOUR[dev], density=True)
ax1.set_xlabel("hellinger_native"); ax1.set_ylabel("density")
ax1.legend(frameon=False); ax1.set_title("Where the two devices land")

s = df.sample(min(40_000, len(df)), random_state=42)
for dev in (SC, ION):
    d = s[s["device"] == dev]
    ax2.scatter(d["hellinger_logical"], d["hellinger_native"], s=2, alpha=0.2,
                color=COLOUR[dev], label=NAME[dev])
ax2.plot([0, 1], [0, 1], color="#718096", lw=0.8, ls="--")
ax2.set_xlabel("hellinger_logical"); ax2.set_ylabel("hellinger_native")
ax2.set_title("The two targets agree (r = 0.999)")
fig.tight_layout()

## 4. Circuit structure explains most of it

Two-qubit gates dominate. This is the signal the circuit-only baseline (Tier 3)
picks up, and the bar that sensor and calibration features fail to clear.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
for dev in (SC, ION):
    d = df[df["device"] == dev]
    grouped = d.groupby(pd.cut(d["native_n_2q_gates"], bins=[0, 5, 10, 20, 40, 80, 160, 1e9]),
                        observed=True)["hellinger_native"].mean()
    ax1.plot(range(len(grouped)), grouped.values, marker="o", ms=4,
             color=COLOUR[dev], label=NAME[dev])
ax1.set_xticks(range(7))
ax1.set_xticklabels(["1-5", "6-10", "11-20", "21-40", "41-80", "81-160", "160+"], rotation=30)
ax1.set_xlabel("two-qubit gates (native)"); ax1.set_ylabel("mean hellinger_native")
ax1.legend(frameon=False); ax1.set_title("More entangling gates, worse fidelity")

q = df[df["device"] == SC]
pivot = q.pivot_table(values="hellinger_native", observed=True,
                      index=pd.cut(q["native_depth"], bins=[0, 20, 50, 100, 200, 500, 1e9]),
                      columns=pd.cut(q["native_n_2q_gates"], bins=[0, 5, 20, 50, 150, 1e9]),
                      aggfunc="mean")
im = ax2.imshow(pivot.values, cmap="magma_r", aspect="auto", origin="lower")
ax2.set_xticks(range(pivot.shape[1])); ax2.set_xticklabels(["1-5", "6-20", "21-50", "51-150", "150+"])
ax2.set_yticks(range(pivot.shape[0])); ax2.set_yticklabels(["1-20", "21-50", "51-100", "101-200", "201-500", "500+"])
ax2.set_xlabel("two-qubit gates"); ax2.set_ylabel("circuit depth")
ax2.set_title("Superconducting: depth x entanglement"); ax2.grid(False)
fig.colorbar(im, ax=ax2, label="mean hellinger_native")
fig.tight_layout()

## 5. Shot count is a first-order driver

Each shot bucket has its own sampling-noise floor (about 1/sqrt(shots)). A model
told the shot count can separate the buckets, and most of the pooled R² comes
from exactly that — which is why the paper reports the single-bucket result as
well.

In [ ]:
shots = (df.groupby(["device", "shots"]).size()
           .rename("records").reset_index().sort_values("records", ascending=False))
top = shots.head(8).copy()
top["share_%"] = (100 * top["records"] / len(df)).round(2)
top["noise_floor"] = (1 / top["shots"] ** 0.5).round(4)
top.reset_index(drop=True)

## 6. Correlation is not contribution

The qubit coherence times correlate with fidelity about as strongly as anything
in the dataset — T2 reaches r = -0.46, and the sign is the physical one: longer
coherence, better runs.

Yet adding all 100 calibration columns to a model that already knows the circuit
raises R^2 by less than a thousandth (section 7). Both facts are true at once: the
calibration describes the machine's condition, and that condition is already
implied by what the circuit had to do and when it ran. This gap between marginal
correlation and incremental contribution is the reason the paper reports DeltaR^2
rather than correlations or SHAP shares.

In [ ]:
q = df[df["device"] == SC]
have = [c for c in sensor_cols + calib_cols if q[c].notna().mean() > 0.3]
corr = (q[have + ["hellinger_native"]].corr(numeric_only=True)["hellinger_native"]
        .drop("hellinger_native").dropna().sort_values())
show = pd.concat([corr.head(8), corr.tail(8)])

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(range(len(show)), show.values,
        color=["#c05621" if v < 0 else "#2b6cb0" for v in show.values])
ax.set_yticks(range(len(show)))
ax.set_yticklabels([c[-42:] for c in show.index], fontsize=7)
ax.set_xlabel("Pearson r with hellinger_native")
ax.set_title("Superconducting device: strongest sensor and calibration correlations")
fig.tight_layout()
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(3)
print("Strongest marginal correlations:")
for name, r in top.items():
    print(f"  {name:<24} r = {r:+.3f}")
print()
print("Section 7 shows what they add to a model that already knows the circuit.")

## 7. The model results

Read from `experiments/results/`, written by `scripts/reproduce_all.sh`. If these
files are missing, run that script first.

`task9_split_summary.csv` is the one to read first: it reports every ΔR² as a
mean over five group splits with its spread, which is the only way to see whether
an effect of a few thousandths is real.

In [ ]:
for name, caption in [("task3_per_device.csv", "Per-device tiers"),
                      ("task9_split_summary.csv", "DeltaR2 over five splits"),
                      ("pool_comparison.csv", "Analysis sets side by side")]:
    path = RESULTS / name
    print(f"\n--- {caption}  ({name}) ---")
    if path.exists():
        print(pd.read_csv(path).to_string(index=False))
    else:
        print("not found; run scripts/reproduce_all.sh")

---

**Where to go next**

- `README.md` — how to reproduce every table in the paper
- `experiments/` — the models themselves
- `qstd_v1.0_columns.csv`, shipped with the dataset — every column, its unit and
  how often it is filled